In [7]:
import os
import time
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from PIL import Image

# =============== 기본 경로 및 모델 로드 ===============
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

models_dir = os.path.join(base_dir, "models")

# -- 분류 모델 --
colors_11n_cls = YOLO(os.path.join(models_dir, "colors_classification_11n_v4.pt"))
colors_11s_cls = YOLO(os.path.join(models_dir, "colors_classification_11s_v1.pt"))

# -- 객체 탐지 모델 --
fruits_11n = YOLO(os.path.join(models_dir, "fruits_detection_11n_v3.pt"))
fruits_11s = YOLO(os.path.join(models_dir, "fruits_detection_11s_v1.pt"))
animals_11n = YOLO(os.path.join(models_dir, "animals_detection_11n_v3.pt"))
animals_11s = YOLO(os.path.join(models_dir, "animals_detection_11s_v1.pt"))

# -- 샘플 이미지 폴더 --
colors_sample_dir = os.path.join(base_dir, "colors_sample")
fruits_sample_dir = os.path.join(base_dir, "fruits_sample")
animals_sample_dir = os.path.join(base_dir, "animals_sample")


# =============== 유틸 함수 ===============
def read_image_cv2(image_path: str) -> np.ndarray:
    """
    파일 경로로부터 OpenCV 이미지(ndarray)를 로드합니다.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"이미지를 불러올 수 없습니다: {image_path}")
    return img


def classify_center_region(model, image_cv2, resize: int = 224, threshold: float = 0.2) -> list:
    """
    이미지 중앙 영역(높이, 너비의 25% ~ 75%)을 잘라 리사이즈 후 분류 모델 예측을 수행합니다.
    반환 예시: [{"prediction_result": 결과, "confidence": 신뢰도}, ...]
    """
    h, w, _ = image_cv2.shape
    center_crop = image_cv2[h // 4: 3 * h // 4, w // 4: 3 * w // 4, :]

    if resize:
        center_crop = cv2.resize(center_crop, (resize, resize), interpolation=cv2.INTER_AREA)

    results = model.predict(source=center_crop, conf=threshold)
    predictions = []
    if results and hasattr(results[0], "probs") and results[0].probs is not None:
        probs_tensor = results[0].probs.data  # torch.Tensor
        k = min(5, probs_tensor.size(0))  # 최대 5개 예측
        topk_values, topk_indices = torch.topk(probs_tensor, k=k)
        for i in range(k):
            conf_val = float(topk_values[i])
            if conf_val >= threshold:
                idx = int(topk_indices[i])
                result_name = (
                    results[0].names[idx]
                    if hasattr(results[0], "names") and results[0].names
                    else str(idx)
                )
                predictions.append({
                    "prediction_result": result_name,
                    "confidence": round(conf_val, 3)
                })
    return predictions


def obj_detection(model, image_cv2, conf_thres: float = 0.25,
                  resize_width: int = 640, resize_height: int = 640):
    """
    객체 탐지를 수행한 후,
      1. 바운딩 박스가 그려진 결과 이미지 (OpenCV ndarray)
      2. 바운딩 박스 정보 리스트 (dict)를 반환합니다.
    """
    resized_img = cv2.resize(image_cv2, (resize_width, resize_height), interpolation=cv2.INTER_AREA)
    results = model.predict(source=resized_img, conf=conf_thres)
    detections = []
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # 좌표, 신뢰도, 클래스 아이디 추출
            x1, y1, x2, y2 = box.xyxy[0].int().tolist()
            conf = box.conf[0].item()
            cls_id = int(box.cls[0].item()) if box.cls is not None else -1

            if hasattr(model, "names") and model.names and cls_id in model.names:
                prediction_result = model.names[cls_id]
            else:
                prediction_result = str(cls_id)

            detections.append({
                "prediction_result": prediction_result,
                "class_id": cls_id,
                "confidence": round(float(conf), 3),
                "bounding_box_area": [x1, y1, x2, y2],
            })
    return resized_img, detections


# =============== 평가 함수 ===============
def evaluate_classification_model(model, sample_folder: str, model_name: str):
    """
    주어진 분류 모델에 대해, sample_folder 내의 이미지들에 대한 예측 및
    예측에 걸린 시간과 예측 결과를 출력합니다.
    """
    image_files = [
        f for f in os.listdir(sample_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]
    if not image_files:
        print(f"[{model_name}] {sample_folder} 폴더에 이미지가 없습니다.")
        return

    total_time = 0.0
    results_summary = []

    for image_file in image_files:
        image_path = os.path.join(sample_folder, image_file)
        try:
            img_cv2 = read_image_cv2(image_path)
        except Exception as e:
            print(f"[{model_name}] 이미지 로드 실패 ({image_path}): {e}")
            continue

        start_time = time.time()
        predictions = classify_center_region(model, img_cv2)
        elapsed = time.time() - start_time
        total_time += elapsed

        results_summary.append({
            "image": image_file,
            "runtime_sec": elapsed,
            "predictions": predictions
        })

    avg_time = total_time / len(results_summary)
    print(f"\n=== 분류 모델 [{model_name}] 평가 ===")
    print(f"총 {len(results_summary)}개 이미지 처리, 평균 예측시간: {avg_time:.4f} 초")
    for r in results_summary:
        print(f"- 이미지: {r['image']}, 시간: {r['runtime_sec']:.4f} 초, 예측: {r['predictions']}")


def evaluate_detection_model(model, sample_folder: str, model_name: str):
    """
    주어진 객체 탐지 모델에 대해, sample_folder 내의 이미지들에 대한 예측 및
    예측에 걸린 시간과 탐지 결과를 출력합니다.
    """
    image_files = [
        f for f in os.listdir(sample_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]
    if not image_files:
        print(f"[{model_name}] {sample_folder} 폴더에 이미지가 없습니다.")
        return

    total_time = 0.0
    results_summary = []

    for image_file in image_files:
        image_path = os.path.join(sample_folder, image_file)
        try:
            img_cv2 = read_image_cv2(image_path)
        except Exception as e:
            print(f"[{model_name}] 이미지 로드 실패 ({image_path}): {e}")
            continue

        start_time = time.time()
        _, detections = obj_detection(model, img_cv2)
        elapsed = time.time() - start_time
        total_time += elapsed

        results_summary.append({
            "image": image_file,
            "runtime_sec": elapsed,
            "detections": detections
        })

    avg_time = total_time / len(results_summary)
    print(f"\n=== 객체 탐지 모델 [{model_name}] 평가 ===")
    print(f"총 {len(results_summary)}개 이미지 처리, 평균 예측시간: {avg_time:.4f} 초")
    for r in results_summary:
        print(f"- 이미지: {r['image']}, 시간: {r['runtime_sec']:.4f} 초, 탐지 결과: {r['detections']}")



In [8]:
print(">>> Colors Classification 모델 성능 평가")
evaluate_classification_model(colors_11n_cls, colors_sample_dir, "colors_11n_cls")
evaluate_classification_model(colors_11s_cls, colors_sample_dir, "colors_11s_cls")


>>> Colors Classification 모델 성능 평가

0: 224x224 blue 0.94, navy 0.05, yellow 0.01, purple 0.00, orange 0.00, 4.5ms
Speed: 3.0ms preprocess, 4.5ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 blue 1.00, yellow 0.00, navy 0.00, purple 0.00, orange 0.00, 9.2ms
Speed: 1.2ms preprocess, 9.2ms inference, 0.5ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 red 0.77, purple 0.23, green 0.01, yellow 0.00, orange 0.00, 9.7ms
Speed: 2.0ms preprocess, 9.7ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 purple 0.62, red 0.29, green 0.09, yellow 0.00, orange 0.00, 6.5ms
Speed: 2.0ms preprocess, 6.5ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 white 0.54, gray 0.18, navy 0.09, orange 0.07, blue 0.05, 7.0ms
Speed: 2.5ms preprocess, 7.0ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 white 0.58, orange 0.19, gray 0.16, yellow 0.04, purple 0.01, 7.5ms
Speed: 2.0ms 

In [9]:
print(">>> Fruits Detection 모델 성능 평가")
evaluate_detection_model(fruits_11n, fruits_sample_dir, "fruits_11n")
evaluate_detection_model(fruits_11s, fruits_sample_dir, "fruits_11s")


>>> Fruits Detection 모델 성능 평가

0: 640x640 1 apple, 11.1ms
Speed: 3.0ms preprocess, 11.1ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 apples, 12.2ms
Speed: 1.0ms preprocess, 12.2ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 apple, 11.0ms
Speed: 1.6ms preprocess, 11.0ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 bananas, 15.6ms
Speed: 2.5ms preprocess, 15.6ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 28 bananas, 11.5ms
Speed: 3.0ms preprocess, 11.5ms inference, 2.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 banana, 3 tomatos, 10.8ms
Speed: 2.0ms preprocess, 10.8ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)

=== 객체 탐지 모델 [fruits_11n] 평가 ===
총 6개 이미지 처리, 평균 예측시간: 0.0523 초
- 이미지: apple-001.jpg, 시간: 0.1863 초, 탐지 결과: [{'prediction_result': 'apple', 'class_id': 0, 'confidence': 0.949, 'bounding_box_area': [69, 

In [10]:
print(">>> Animals Detection 모델 성능 평가")
evaluate_detection_model(animals_11n, animals_sample_dir, "animals_11n")
evaluate_detection_model(animals_11s, animals_sample_dir, "animals_11s")


>>> Animals Detection 모델 성능 평가

0: 640x640 1 cat, 11.2ms
Speed: 2.6ms preprocess, 11.2ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 cat, 11.2ms
Speed: 2.0ms preprocess, 11.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 cat, 13.0ms
Speed: 2.0ms preprocess, 13.0ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 dog, 8.1ms
Speed: 4.0ms preprocess, 8.1ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 cows, 8.5ms
Speed: 2.0ms preprocess, 8.5ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 dog, 10.0ms
Speed: 1.1ms preprocess, 10.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

=== 객체 탐지 모델 [animals_11n] 평가 ===
총 6개 이미지 처리, 평균 예측시간: 0.0392 초
- 이미지: cat-001.jpg, 시간: 0.1286 초, 탐지 결과: [{'prediction_result': 'cat', 'class_id': 0, 'confidence': 0.908, 'bounding_box_area': [2, 22, 520, 638]}]
- 이미지: cat-002.jpg, 